# Tái lập Bảng 1 của SAGE: SegFormer-B3 trên FMB

Mục tiêu: train SegFormer-B3 trên từng loại ảnh (chỉ VIS, chỉ IR, ảnh ghép SAGE, ghép trung bình) rồi chấm mIoU trên FMB test, để so với Bảng 1 của bài (VIS 49,1; IR 42,7; SAGE 61,2 – theo bài).

**Chuẩn bị (làm một lần):**
1. *Add Input* → dataset **FMB** (thư mục có `train/Visible`, `train/Infrared`, `train/Label`, `test/...`).
2. *Add Input* → dataset **code**: một thư mục gồm các file `src/segmentation/*.py` của repo, thư mục `manifests/` (chép từ `data/manifests/`) và `sage_student/` (`model.py` + `checkpoints.pt` của SAGE Student). Bản đã dùng cho kết quả khóa 14/11: dataset riêng tư `thanhmn123/sage-tai-lap`.
3. *Settings → Accelerator* → **GPU T4 x2**. *Internet* → **On** (để tải trọng số MiT-B3).

**Cách chạy:**
- Lần đầu để `THU = True`, bấm *Run All*. Mất vài phút. Đọc dòng "Tốc độ … | bộ nhớ … | Ước tính …".
- Sau đó đặt `THU = False`, bấm **Save Version → Save & Run All (Commit)**. Kaggle chạy nền, tắt máy vẫn chạy.

In [ ]:
# ===== CẤU HÌNH =====
THU = False                   # True: chạy thử để đo tốc độ, bộ nhớ.  False: chạy thật
BIEN_THE = ["sage", "vis"]    # chọn trong: vis, ir, tb, sage, clahe. Mỗi GPU chạy một biến thể cùng lúc
SEED = [2]
MODEL, EPOCHS = "b3", 100     # bài: SegFormer-B3, 100 epoch
BATCH, TICH_LUY = 8, 1        # nếu báo hết bộ nhớ (out of memory): BATCH, TICH_LUY = 4, 2
GOP_DEV = True                # True: train đủ 1.220 ảnh train chính thức như bài, chấm last.pt
                              # False: train 1.045 ảnh, chọn checkpoint bằng 175 ảnh dev (giao thức sạch của nhóm)
LOP0 = True                   # True: unlabeled là một lớp (model 15 lớp), như code SegMiF cùng nhóm tác giả.
                              #       Chấm ra được cả mIoU 15 lớp lẫn mIoU bỏ unlabeled
                              # False: bỏ qua pixel unlabeled khi train (model 14 lớp)

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if KAGGLE:
    nguon = [p.parent for p in Path("/kaggle/input").rglob("train_seg.py")]
    assert nguon, "Chưa thêm dataset code (sage-tai-lap) vào Input"
    CODE = Path("/kaggle/working/code")
    shutil.copytree(nguon[0], CODE, dirs_exist_ok=True)
    os.environ["SAGE_ANH"] = "/kaggle/temp/anh"         # ảnh đầu vào: không cần giữ lại
    os.environ["SAGE_RUNS"] = "/kaggle/working/runs"    # kết quả: Kaggle giữ lại sau khi Commit
    # Nếu Input có kết quả của lần chạy trước (thư mục runs/...), chép sang để bỏ qua việc đã xong / train tiếp
    for c in Path("/kaggle/input").rglob("runs/*/config.json"):
        dich = Path(os.environ["SAGE_RUNS"]) / c.parent.name
        if not dich.exists():
            shutil.copytree(c.parent, dich)
            print("chép run cũ:", c.parent.name)
else:
    CODE = Path.cwd()
os.environ["PYTHONIOENCODING"] = "utf-8"
os.chdir(CODE)
sys.path.insert(0, str(CODE))

import pandas as pd, torch
import chung

print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "Chưa bật GPU (Settings → Accelerator → GPU T4 x2)"
print("FMB:", {k: str(v) for k, v in chung._tim_fmb().items()})
n = {s: len(pd.read_csv(chung.MAN / f"{s}.csv")) for s in ("train", "dev", "test")}
print("Manifest:", n)
assert n == {"train": 1045, "dev": 175, "test": 280}


def chay(*lenh):
    print(">", " ".join(map(str, lenh)), flush=True)
    p = subprocess.Popen([sys.executable, *map(str, lenh)], cwd=CODE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for d in p.stdout:
        print(d, end="")
    assert p.wait() == 0, "Lệnh lỗi, xem thông báo ở trên"

## 1. Xuất ảnh đầu vào

`sage` = chạy Student pretrained của SAGE trên 1.500 cặp. `vis`, `ir`, `tb` không cần mạng.

In [ ]:
for bt in (BIEN_THE[:1] if THU else BIEN_THE):
    chay("xuat_anh.py", "--bien_the", bt)

## 2. Train và chấm

Khi chạy thật, cứ 10 phút in một dòng tiến độ cho mỗi biến thể. Log đầy đủ nằm ở `runs/log_*.txt`.

In [ ]:
chung_ts = ["--model", MODEL, "--epochs", EPOCHS, "--batch", BATCH, "--tich_luy", TICH_LUY] + (["--gop_dev"] if GOP_DEV else []) + (["--lop0"] if LOP0 else [])
if THU:
    chay("train_seg.py", "--bien_the", BIEN_THE[0], "--thu", 40, *chung_ts)
    print("\nĐây là chạy thử. Đặt THU = False rồi Save Version → Save & Run All để chạy thật.")
else:
    chay("chay_song_song.py", "--bien_the", *BIEN_THE, "--seed", *SEED, *chung_ts)

## 3. Bảng kết quả

Bảng cuối đặt số của nhóm cạnh số của bài. Chỉ **7 lớp bài in ra** là so trực tiếp được; bài không nói mIoU tính trên lớp nào.

In [ ]:
if not THU:
    chay("tong_hop.py")

## Sau khi chạy xong

Vào tab **Output** của notebook, tải về thư mục `runs/` (hoặc ít nhất `tong_hop.csv`, các file `test_ket_qua_last.json`, `test_iou_lop_last.csv`, `test_nham_last.npz`, `log.csv`).

Muốn chạy thêm biến thể (ví dụ `tb`, `ir`) mà vẫn giữ kết quả cũ trong bảng: *Add Input* → chọn Output của lần chạy trước, sửa `BIEN_THE`, rồi Commit lại.